In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
os.chdir('/home/jhpark/image-artifacts/src')
os.environ["CUDA_VISIBLE_DEVICES"] = "2"


In [4]:
# Install required packages if needed
# !pip install torch torchvision transformers openai PIL numpy tqdm matplotlib
# !pip install groundingdino-py segment-anything

import sys
import uuid
import json
import time
import logging
import pickle
import random
import shutil
from datetime import datetime
from typing import List, Dict, Optional, Tuple, Any
import traceback
from collections import defaultdict

import numpy as np
import openai
import torch
from tqdm import tqdm
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import matplotlib.pyplot as plt

# Add pipeline to path

from pipeline import (
    GSAMDetector, InstanceProcessor, ImageVisualizer,
)
from pipeline.data_loader import _initialize_data_loader, _get_image_list
from pipeline.prompts import get_entity_subentities, MoneyManager

/home/jhpark/anaconda3/envs/gsam/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Configuration - modify these as needed
CONFIG = {
    # Dataset configuration
    'dataset_type': 'coco',
    'dataset_path': '/data3/jhpark/coco/annotations',
    'image_path': '/data3/jhpark/coco/train2017',
    'super_categories': ['animal'],  # Process animal images
    
    # Processing parameters
    'max_images': 1,  # Process only one image
    'max_artifacts_per_image': 2,  # Generate 2 artifacts per image
    'artifact_types': ['fusion', 'distortion', 'removal', 'addition'],
    
    # GSAM parameters
    'min_area_ratio': 0.005,
    'max_area_ratio': 0.5,
    'box_threshold': 0.3,
    'text_threshold': 0.25,
    'device': 'cuda:0' if torch.cuda.is_available() else 'cpu',
    
    # Model paths (adjust if needed)
    'grounding_config_file': 'GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py',
    'grounding_checkpoint': 'weight/groundingdino_swint_ogc.pth',
    'sam_version': 'vit_h',
    'sam_checkpoint': 'weight/sam_vit_h_4b8939.pth',
    'sam_hq_checkpoint': None,
    'use_sam_hq': False,
    'bert_base_uncased_path': None,
    
    # FLUX parameters
    'guidance': 5.0,
    'num_steps': 25,
    'inject_step': 20,
    'pe_step_addition': 25,
    'pe_step_removal': 25,
    'pe_step_distortion': 20,
    'pe_step_fusion': 20,
    'seed': 42,
    'use_rf_solver': False
}

# Set random seed for reproducibility
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration loaded:
  dataset_type: coco
  dataset_path: /data3/jhpark/coco/annotations
  image_path: /data3/jhpark/coco/train2017
  super_categories: ['animal']
  max_images: 1
  max_artifacts_per_image: 2
  artifact_types: ['fusion', 'distortion', 'removal', 'addition']
  min_area_ratio: 0.005
  max_area_ratio: 0.5
  box_threshold: 0.3
  text_threshold: 0.25
  device: cuda:0
  grounding_config_file: GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py
  grounding_checkpoint: weight/groundingdino_swint_ogc.pth
  sam_version: vit_h
  sam_checkpoint: weight/sam_vit_h_4b8939.pth
  sam_hq_checkpoint: None
  use_sam_hq: False
  bert_base_uncased_path: None
  guidance: 5.0
  num_steps: 25
  inject_step: 20
  pe_step_addition: 25
  pe_step_removal: 25
  pe_step_distortion: 20
  pe_step_fusion: 20
  seed: 42
  use_rf_solver: False


In [6]:
# Initialize OpenAI client
if not os.getenv('OPENAI_API_KEY'):
    print("❌ Error: OPENAI_API_KEY environment variable not set.")
    print("Please set your OpenAI API key:")
    print("  export OPENAI_API_KEY='your-api-key-here'")
else:
    openai_client = openai.OpenAI()
    print("✅ OpenAI client initialized")

# Initialize MoneyManager for tracking API costs
money_manager = MoneyManager(model="gpt-4o")

✅ OpenAI client initialized


In [7]:
data_loader = _initialize_data_loader(CONFIG['dataset_type'], CONFIG)


loading annotations into memory...
Done (t=0.86s)
creating index...
index created!
loading annotations into memory...
Done (t=16.53s)
creating index...
index created!


In [11]:
if CONFIG['dataset_type'].lower() == 'coco':
    supercategories = set()
    for cat in data_loader.coco_class.dataset['categories']:
        supercategories.add(cat['supercategory'])
    print("COCO supercategories:", sorted(supercategories))

data_num = {}
for sup_cat in supercategories:
    image_list = _get_image_list(
        CONFIG['dataset_type'], 
        data_loader, 
        [sup_cat], 
        max_instances_per_image=3,
        logger=logging.getLogger()
    )
    data_num[sup_cat] = len(image_list)



COCO supercategories: ['accessory', 'animal', 'appliance', 'electronic', 'food', 'furniture', 'indoor', 'kitchen', 'outdoor', 'person', 'sports', 'vehicle']
Counting instances per image...
number of images 957
Counting instances per image...
number of images 2780
Counting instances per image...
number of images 9814
Counting instances per image...
number of images 1348
Counting instances per image...
number of images 1568
Counting instances per image...
number of images 5691
Counting instances per image...
number of images 1787
Counting instances per image...
number of images 4159
Counting instances per image...
number of images 3660
Counting instances per image...
number of images 2523
Counting instances per image...
number of images 10447
Counting instances per image...
number of images 6122


In [12]:
sum(data_num.values())

50856

{'kitchen': 957, 'outdoor': 2780, 'animal': 9814, 'electronic': 1348, 'accessory': 1568, 'vehicle': 5691, 'appliance': 1787, 'furniture': 4159, 'indoor': 3660, 'food': 2523, 'person': 10447, 'sports': 6122}
